# ChronoMind ReVA temporal QA baseline
Uses frozen CLIP encoders and a trainable temporal/fusion head. No QA metrics are claimed until this notebook runs on labeled data.
Set ROLE3_QA_REVISION only after the QA implementation has been committed; the placeholder intentionally stops execution. Google Drive is optional; artifacts remain under /content/role3_qa_artifacts.

In [ ]:
import torch, platform
print(platform.python_version(), torch.__version__, torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0), torch.cuda.mem_get_info())

In [ ]:
%pip -q install transformers huggingface_hub opencv-python-headless
# Colab provides PyTorch; no local model weights are bundled.

In [ ]:
import subprocess, os, json
ROLE3_QA_REVISION = "ceec2dc8c25c2b9961260b8e0e95dd40fcf54236"
SARTHAK_REVISION = "1a63b5009267a348b5fdd550b063869332d4ce0a"
DATASET_REVISION = "06224fa1b42e5e35677a8b7f2145e06307a9a4ee"
if ROLE3_QA_REVISION == "SET_AFTER_QA_COMMIT" or ROLE3_QA_REVISION == "62f383676b63717702ae27eb52e023a74c2e1d38": raise ValueError("Set ROLE3_QA_REVISION to the commit containing the QA modules; do not use the old Role 3 revision.")
REPO="https://github.com/sarthak-shastri/ChronoMind.git"
subprocess.run(["git","clone","--quiet",REPO,"/content/ChronoMind"],check=True)
subprocess.run(["git","-C","/content/ChronoMind","checkout","--detach",ROLE3_QA_REVISION],check=True)
subprocess.run(["git","clone","--quiet","--no-checkout",REPO,"/content/ChronoMind-sarthak"],check=True)
subprocess.run(["git","-C","/content/ChronoMind-sarthak","checkout","--detach",SARTHAK_REVISION],check=True)
os.environ['PYTHONPATH']="/content/ChronoMind:/content/ChronoMind-sarthak"
print('Role3',subprocess.check_output(["git","-C","/content/ChronoMind","rev-parse","HEAD"],text=True).strip())
print('Sarthak',subprocess.check_output(["git","-C","/content/ChronoMind-sarthak","rev-parse","HEAD"],text=True).strip())

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path
DATASET_REPO='ReVA-Benchmark/ReVA'
CACHE=Path('/content/reva_cache'); CACHE.mkdir(exist_ok=True)
train_json=hf_hub_download(DATASET_REPO,'train.json',repo_type='dataset',revision=DATASET_REVISION,cache_dir=CACHE)
val_json=hf_hub_download(DATASET_REPO,'val.json',repo_type='dataset',revision=DATASET_REVISION,cache_dir=CACHE)
train_data=json.load(open(train_json)); val_data=json.load(open(val_json))
assert isinstance(train_data.get('QA'),list) and isinstance(val_data.get('QA'),list)
assert all(set(x.get('options',{}))=={'A','B','C','D'} for x in train_data['QA'][:5])
print('QA rows',len(train_data['QA']),len(val_data['QA']),'example',train_data['QA'][0]['qa_id'],train_data['QA'][0]['video_path'])
# Dataset files resolve lazily through Sarthak's ReVADataset; missing videos raise.

In [ ]:
import sys
sys.path.insert(0,'/content/ChronoMind'); sys.path.insert(0,'/content/ChronoMind-sarthak')
from preprocessing import ReVADataset
from advanced_models.qa.text_encoder import CLIPTextFeatureExtractor
from advanced_models.qa.multimodal_qa_model import MultimodalVideoQAModel
from advanced_models.temporal.temporal_encoder import TransformerTemporalEncoder
from advanced_models.vision.feature_extractor import CLIPVisionFeatureExtractor
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
vision=CLIPVisionFeatureExtractor.from_pretrained(input_normalization='imagenet',frame_chunk_size=4,cuda_mixed_precision=device.type=='cuda')
text=CLIPTextFeatureExtractor.from_pretrained()
model=MultimodalVideoQAModel(vision,text,TransformerTemporalEncoder(512,num_heads=8,num_layers=2,max_seq_len=16)).to(device)
print('Frozen encoders loaded:',device)

In [ ]:
from preprocessing import ReVADataset
from advanced_models.qa.data import ReVAQADatasetAdapter,collate_qa
smoke=ReVAQADatasetAdapter(ReVADataset(val_json,num_frames=4,split='val',sample_ids=[val_data['QA'][0]['qa_id']],dataset_repo_id=DATASET_REPO,revision=DATASET_REVISION,missing_video='raise'))
batch=collate_qa([smoke[0]])
with torch.inference_mode(): print('forward logits',model(batch['video_frames'].to(device),batch['questions'],batch['choices'])['answer_logits'].shape)

## Tiny smoke and first experiment
The following cell trains on a small subset only; change MAX_* to None for full official splits after verifying runtime and storage. Baseline comparison can be run with the frozen cosine helper after the baseline script is configured. Training default is 8 frames; 4/16 remain selectable.

In [ ]:
# Tiny real-data smoke train: 16 train / 8 validation samples. Expand limits for the full experiment.
cmd=['python','/content/ChronoMind/scripts/train_reva_qa.py','--train-annotations',train_json,'--val-annotations',val_json,'--dataset-repo-id',DATASET_REPO,'--revision',DATASET_REVISION,'--preprocessing-root','/content/ChronoMind-sarthak','--frames','8','--batch-size','2','--epochs','1','--seed','17','--mixed-precision','--limit-train','16','--limit-val','8','--output','/content/role3_qa_artifacts']
subprocess.run(cmd,check=True)

In [ ]:
# Tiny validation export; omit --limit-samples 8 to evaluate the complete official validation split.
EVAL_CMD=['python','/content/ChronoMind/scripts/evaluate_reva_qa.py','--annotations',val_json,'--checkpoint','/content/role3_qa_artifacts/best.pt','--preprocessing-root','/content/ChronoMind-sarthak','--dataset-repo-id',DATASET_REPO,'--revision',DATASET_REVISION,'--limit-samples','8','--output-dir','/content/role3_qa_artifacts/evaluation']
subprocess.run(EVAL_CMD,check=True)
print('Saved predictions.json and metrics.json under /content/role3_qa_artifacts/evaluation')

In [ ]:
# Optional Google Drive persistence; otherwise download artifacts from Colab Files panel.
PERSIST_TO_DRIVE=False
if PERSIST_TO_DRIVE:
 from google.colab import drive
 drive.mount('/content/drive')
 subprocess.run(['cp','-r','/content/role3_qa_artifacts','/content/drive/MyDrive/'],check=True)